In [ ]:
#1
import numpy as np 
import matplotlib.pyplot as plt 

# Physical Constants
h_bar = 1.0545718e-34
c = 3e8

# Light Source Parameters 
sources = {
    "Violet Laser": {"lambda": 405e-9, "color": 'Violet'},
    "Red Laser": {"lambda": 650e-9, "color": 'red'}
}

# Grid
grid_size = 2048 
physical_width = 0.02
x = np.linspace(-physical_width/2, physical_width/2, grid_size)
dx = x[1] - x[0]
#2
def create_aperature(width_delta_x):
    # Light Blocking
    aperture = np.zeros(grid_size)
    # Groups and Identifies the pixels in the slit convergence point 
    limit = width_delta_x / 2
    aperture[(x > -limit) & (x < limit)] = 1
    return aperture

# The current dx value will be the changable delta x value. 
# This value could be random or could be one from my research in checking the validity. 
current_dx = 2e-5
mask = create_aperature(current_dx) 
#3
def simulate_diffraction(aperture, wavelength):
    # The Fast Fourier Transform
    field_fft = np.fft.fft(aperture)

    # Helps show the peak intensity in the middle
    field_shifted = np.fft.fftshift(field_fft)

    # Defines Instensity 
    intensity = np.abs(field_shifted)**2

    # Normalized intensity to make it easier to understand differnces
    intensity /= np.max(intensity)

    # Finding momentum
    # From equation 2
    frequencies = np.fft.fftfreq(grid_size, d=dx)
    p_x = np.fft.fftshift(frequencies) * 6.626e-34 
    #Conversion to p_x

    return p_x, intensity
#4
def get_uncertainty_product(px_axis, intensity, dx_val):
    # Finds the width a 13.5% (1/e^2 width)
    mask = intensity > 0.135
    delta_px = np.max(px_axis[mask]) - np.min(px_axis[mask])

    product = dx_val * delta_px
    return delta_px, product

# This is the test for the violet laser in example. 
# This part needs to be changed everytime the source is changed.
px, intensity_profile = simulate_diffraction(mask, sources["Violet Laser"]["lambda"])
d_px, uncert_product = get_uncertainty_product(px, intensity_profile, current_dx)

print(f"Uncertainty Product: {uncert_product:.2e} J*s")
print(f"Heisenberg Limit (hbar/2): {h_bar/2:.2e} J*s")
#5 
plt.figure(figsize=(12, 5))

# Position Side
plt.subplot(1, 2, 1)
plt.plot(x * 1000, mask, color='purple') # x is in millimeters
plt.title(f"Position Domain (Slit Width: {current_dx*1e6:1f} µm)")
plt.xlabel("Position (mm)")
plt.ylabel("Aperture Transmission (0 to 1)")
plt.grid(True, alpha=0.3)

# Momentum Side 
plt.subplot(1, 2, 2)
plt.plot(px, intensity_profile, color='red')
#13.5% Threshold
plt.axhline(y=0.135, color='black', linestyle='--', label='13.5% Threshold')
plt.title("Momentum Domain (Diffraction Pattern)")
plt.xlabel(r"Momentum $p_x$ ($kg \cdot m/s$)")
plt.ylabel("Normalized Intensity")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()